# TritFlow：日本初の商用オープンソースPDKで、平衡三値HDLからGDSまで

Einosuke Okazaki<br>
Tokyo Metropolitan College of Industrial Technology (KOSEN), Tokyo, Japan

**実行環境**

- 所要時間：新しいColabのランタイムでRun all、セットアップを含めて約13〜15分
- Python：3.12
- PDK：OpenSUSI TR-1um（commit 5d963db）
- KLayout：0.30.6
- ngspice：42
- Pythonのパッケージ：最初のセルが固定したバージョンで導入

TritFlowは、三つの電圧レベルをそのまま扱うCMOSセルを用い、三値のHDLからGDSまでをOpenSUSI TR-1umの上で通す設計フローである。本ノートブックでは三値のHDLから論理合成、2層メタルの配置配線、GDSの出力、DRC/LVS、ポストレイアウトシミュレーション、三値のタイミング解析までを順に実行する。

**要点**

- **我々の知る限り初の、HDLからGDSまで通す三値フィジカル設計フロー**：三値専用のHDL（TritVerilog）→ 論理合成 → 2層メタル配置配線 → GDS → DRC/LVS → ポストレイアウトSPICE → 三値のタイミング解析を、1つのパイプラインで自動で通す。
- **物理的な平衡三値**：三値を二値に符号化せず、−1・0・+1 を −5 V・0 V・+5 V の3レベルの電圧のまま扱うCMOSセルで回路を組む。
- **製造できるGDS**：PDK公式デッキのDRC/LVSに合格し、MPWシャトルのフレームとパッドを付けたフルチップのGDSまで生成する。
- **テープアウト済みの回路に基づくスタンダードセル**：セルは、著者が2026年にOpenSUSI-TR10でテープアウトした平衡三値論理回路をベースにしている。チップは2027年1月に納品予定で、シミュレーション結果を実測と直接比べられる。
- **Colabですべての工程を対話的に体験できる**：Run allで約13〜15分。各工程の回路図、レイアウト、波形をその場で表示し、HDLを書き換えて再実行できる。Apache-2.0。
- **日本初の商用オープンソースPDK上のフロー**：OpenSUSI TR-1umの上に構築したフローとして、日本における半導体設計の民主化を強く前進させる。

![fig_flow_v2](https://raw.githubusercontent.com/PenguinEino/tritflow/ad168eb75b9755c0694159406eb999a1a291b99b/figures/out/fig_flow_v2_ja.png)

TritFlowが扱うのは**平衡三値**（Balanced Ternary）である。各桁（トリット）は −1・0・+1 の3値をとり、桁の重みは3のべき乗（1, 3, 9, …）となる。回路上の電圧としてはそれぞれ −5 V・0 V・+5 V に対応する。平衡三値は1958年にモスクワ大学で開発された初期の三値計算機「[Setun](https://doi.org/10.1007/978-3-642-22816-2_10)」でも採用された歴史ある表現形式である。近年では重みをこの三値（−1・0・+1）に制約した大規模言語モデル「[BitNet b1.58](https://doi.org/10.48550/arXiv.2402.17764)」の登場により再び大きな注目を集めている。線形層における重みと8ビット入力との積は「入力そのまま」「符号反転」「ゼロ」のいずれかとなるため、乗算器を用いず加減算のみで行列積（GEMM）を実行できる。三値のデータを中間の二値変換を挟まず、三値の電圧のままダイレクトに処理する専用ハードウェアは、高い効率を達成する可能性を秘めている。

しかし現在、三値演算を担っているハードウェアは、三値を二値で符号化して扱う従来の二値論理回路である。デバイスレベルではCNTFET（[Lin et al., 2011](https://doi.org/10.1109/TNANO.2009.2036845)）、メモリスタ（[Wang et al., 2021](https://doi.org/10.1109/TCSI.2020.3027693)）、新規CMOS構造（[Jeong et al., 2019](https://doi.org/10.1038/s41928-019-0272-8)）など多様な三値素子が研究されているものの、設計ツール環境は[ゲート回路網の生成](https://doi.org/10.1109/ACCESS.2025.3597293)にとどまるか、あるいは三値信号を複数の二値信号線に符号化して二値EDAを通すアプローチ（[MRCS](https://github.com/aiunderstand/MixedRadixCircuitSynthesis/tree/0865cee1f49fe9a803b4910b570f4e96e514c61b)、[Takahe](https://github.com/Zaneham/takahe/blob/e7123ea60269922acb283d576678a3aabe51f4f4/designs/setun70_physical.sv)、[Ankhdjet](https://doi.org/10.48550/arXiv.2608.26206)）に限られていた。商用ファブで製造可能なオープンソースPDK上で、3つの物理電圧レベルを直接扱うCMOSセルを用いて、HDLからGDSまでを完全自動で通すフィジカル設計フローは、我々の知る限り前例がない。

本ノートブックでは、日本で初めて商用の半導体ファブが公開したオープンソースPDK「[OpenSUSI TR-1um](https://github.com/OpenSUSI/TR-1um/tree/5d963db9d44e192d37a0489c208bfccf2e2f690c)（東海理化）」を採用し、この三値設計フローを実証する。TritFlowのセルライブラリは、[著者が設計・テープアウトした三値論理回路](https://github.com/PenguinEino/opensusi-tr10-balanced-ternary-logic/tree/8a60b97868cad008ff21fefca5da13dcd06063f8)（2026年、第1回OpenSUSI-MPWシャトル「OpenSUSI-TR10」）の回路トポロジーと素子寸法をそのままスタンダードセル化したものである。三値専用HDL「TritVerilog」の記述から論理合成、配置配線、DRC/LVS検証、ポストレイアウトSPICEシミュレーションまでを1つのパイプラインとして完結させている。

OpenSUSI-TR10で製造した実チップは2027年1月に納品予定であり、本提案が採択された暁には、2月のISSCC会場においてTritFlowのデモとともに、三値セルの実測結果を報告する。さらにTritFlowを用いて設計した三値論理回路は、次回のOpenSUSIシャトルにて製造を行う。「設計・製造・実測・学会報告」のサイクルを一気通貫で完結させることで、日本初の商用オープンソースPDKであるTR-1umがIEEE SSCS Code-a-Chipの掲げる価値を実現するに足るポテンシャルを持つことを証明し、日本におけるオープン半導体エコシステムと設計の民主化を強く前進させる。

**再現性**（Reproducibility）　後に続く研究者や技術者、そして学習者がTritFlowの動作原理を隅々まで理解し、自らの手で再構築・検証できるように設計されている。そのために、セルの内部回路、論理合成アルゴリズム、配置配線の手法はいずれも、コードを読んで明快に追える直感的な見通しの良さを重視した。同時に、PDK公式のDRC/LVSデッキを完全にパスし、ポストレイアウトシミュレーションで動作確認された実際に製造可能なGDSを出力する。ノートブック内の各セルは各処理段が「なぜそのように動作するのか」を物理的・理論的な背景とともに説明し、読者がその場でパラメータを変えて即座に検証できるように構成されている。さらに、セルの回路と素子寸法はテープアウトした回路と同じであるため、シミュレーション結果を将来の実測データと直接突き合わせることができる。

**再利用性**（Reusability）　ノートブックで得た理解をもとに、TritFlowを土台として読者独自の研究や設計へと容易に発展させることができる。TritVerilogの記述を書き換えるだけで、同一の自動フローによって検証済みの新しい三値コアレイアウトを再生成できる。後半では4項の内積回路をMPWシャトルの標準フレームおよび入出力パッドと結合したフルチップ・レイアウトまで自動生成する。すべてのアルゴリズムが明瞭なPythonモジュールとして分割されているため、新しい三値ゲートの追加や、論理合成・配置配線アルゴリズムの拡張も自在である。セルライブラリ、EDAパイプライン、本ノートブックはすべてオープンソース（Apache-2.0）で公開されており、三値論理研究の1つの出発点となり得る。

**教育性**（Educational Value）　この「理解から発展へ」の道筋は、そのまま次代を担う学生やエンジニアにとって最高の教材となる。すべての工程がGoogle Colab上で対話的に実行され、各段階で回路網やレイアウト、電圧波形がどのように変化したのかを豊かな可視化（回路図、レイアウト、波形）と詳細な数値データで示す。二値CMOSの基本とVerilogの初歩を理解している読者であれば、HDLの1行から始めてシリコン製造用GDSが完成するまでの全貌を、自らの手を動かしながら体系的に体験できる。単なる理論のシミュレーションにとどまらず、実際に製造可能なGDSが得られ、実測に直結した集積回路設計を体験できる。

In [ ]:
#@title 準備 { display-mode: "form" }

import os, sys, json, time, subprocess, importlib.util, re
from pathlib import Path
SOURCE_URL = "https://github.com/PenguinEino/tritflow.git"
SOURCE_REF = os.environ.get("TRITFLOW_REF", "ad168eb75b9755c0694159406eb999a1a291b99b")
try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False

def git(*args, progress=None):
    """Run git; with ``progress``, show git's own counter on one redrawn line."""
    if progress is None:
        return subprocess.check_output(["git", "-C", str(ROOT), *args], text=True).strip()
    from IPython.display import display, HTML
    style = "font:13px/1.6 ui-monospace,Menlo,Consolas,monospace;white-space:pre"
    line = display(HTML(f'<div style="{style}">⠿ {progress}</div>'), display_id=True)
    process, text, started = subprocess.Popen(["git", *args], stderr=subprocess.PIPE, text=True), "", time.monotonic()
    for chunk in iter(lambda: process.stderr.read(64), ""):
        text += chunk
        parts = [p.strip() for p in re.split(r"[\r\n]", text)[:-1] if p.strip()]
        if parts:
            line.update(HTML(f'<div style="{style}">⠿ <b>{progress}</b>  <span style="color:#8a8f98">{parts[-1][:80]}</span></div>'))
    if process.wait():
        raise RuntimeError(f"git {' '.join(args[:2])} failed:\n{text[-2000:]}")
    line.update(HTML(f'<div style="{style}"><span style="color:#1a9c4a">✓</span> <b>{progress}</b>  '
                     f'<span style="color:#8a8f98">{time.monotonic()-started:.1f}s</span></div>'))

if IN_COLAB:
    ROOT = Path("/content/tritflow")
    if not re.fullmatch(r"[0-9a-f]{40}", SOURCE_REF):
        raise ValueError("TRITFLOW_REF must be a full 40-character commit")
    if not ROOT.exists():
        git("clone", "--progress", "--no-checkout", "--depth", "1", "--filter=blob:none",
            SOURCE_URL, str(ROOT), progress="GitHubからソースを取得")
        git("-C", str(ROOT), "fetch", "--progress", "--depth", "1", "origin", SOURCE_REF,
            progress=f"コミット {SOURCE_REF[:7]} を取得")
        git("-C", str(ROOT), "switch", "--detach", SOURCE_REF, progress="ファイルを展開")
    if git("rev-parse", "HEAD") != SOURCE_REF or git("status", "--porcelain", "--untracked-files=no"):
        # A runtime left by another version of this notebook: move it to the pinned commit and
        # drop the modules the kernel already imported from the old checkout.
        git("-C", str(ROOT), "fetch", "--progress", "--depth", "1", "origin", SOURCE_REF,
            progress=f"コミット {SOURCE_REF[:7]} に切り替え")
        git("reset", "--hard", SOURCE_REF)
        for name, module in list(sys.modules.items()):
            if str(getattr(module, "__file__", "") or "").startswith(str(ROOT)):
                del sys.modules[name]
else:
    candidates = [Path(os.environ.get("TRITFLOW_ROOT", ".")), Path.cwd(), Path.cwd().parent]
    ROOT = next((p.resolve() for p in candidates if (p/"tritflow").is_dir() and (p/"notebook").is_dir()), None)
    if ROOT is None:
        raise RuntimeError("run from the TritFlow checkout or set TRITFLOW_ROOT")
sys.path[:0] = [str(ROOT), str(ROOT/"notebook")]
from live import step, run_stages, set_language as set_notebook_language
set_notebook_language("ja")
from runtime_support import SETUP_SCHEMA, setup_identity, check_runtime, print_tool_versions
runtime = ROOT/"notebook/runtime.json"
if IN_COLAB and (not runtime.is_file()
                 or json.loads(runtime.read_text()).get("TRITFLOW_SETUP_SHA256") != setup_identity(ROOT)):
    os.environ["TRITFLOW_NOTEBOOK_LANGUAGE"] = "ja"
    run_stages(["bash", str(ROOT/"notebook/colab_setup.sh"), "--source", str(ROOT)])
if runtime.is_file():
    os.environ.update(json.loads(runtime.read_text()))
    sys.path.insert(0, os.environ["TRITFLOW_PYTHON_TARGET"])
os.environ.setdefault("QT_QPA_PLATFORM", "offscreen")
with step("ツールを確認") as s:
    check_runtime()
    tools = print_tool_versions()
    s.done(" · ".join(re.search(r"(KLayout [\d.]+|ngspice-\d+)", v)[1] for v in tools.values()))
BUILD = Path(os.environ.get("TRITFLOW_NOTEBOOK_BUILD", "/content/build" if IN_COLAB else ROOT/"build/notebook")).resolve()

from live import need, provides, install_error_view, LayoutView
from views import pick_input, stimulus, bus_picker, cell_counts, schematic_view, scope_figure, timing_table, wire_table, delay_table, transient_conditions, dc_conditions, note
from IPython.display import HTML, display
import plotly.io as pio
pio.renderers.default = "colab"
from inverter_views import inverter
from pnr_views import replay
from cell_views import closure_view, cell_tables, cell_row
from tritflow import CELLS_DIR
from tritflow.sim.sizing import sweep_dimensions, load_grid
from tritflow import flow
from tritflow.viz.layout import gds_viewer
from tritflow.schematic import set_language
set_language("ja")
install_error_view()
for name, cell in {"source": "1-1 HDL", "net": "1-2 合成する", "lay": "1-3 配置配線する",
                   "lvs": "1-4 DRCとLVS", "post": "1-5 ポストレイアウトシミュレーション",
                   "dot4": "3-8 論理合成する", "dot4_layout": "4-1 配置配線する",
                   "dot4_lvs": "5-1 DRCとLVS", "dot4_extracted": "5-2 抽出と配線容量", "dot4_post": "5-3 ポストレイアウトシミュレーション",
                   "dot_scaling": "6-1 内積の規模と面積",
                   "reuse_source": "6-4 HDLを変更する"}.items():
    provides(name, cell)


## 1. 最初の回路：`a & ~a` をGDSにする

本章では三値論理の特性を表しつつもコンパクトな論理回路である `a & ~a` を題材に、TritFlowの全体フロー（HDL記述から論理合成、配置配線、DRC/LVS、ポストレイアウトSPICEシミュレーションまで）を一気通貫で体験する。各処理段の詳細な仕組みは2章以降で順を追って解説する。

二値論理において、入力 a とその反転の論理積 `a & ~a` は常に偽（0）となり、回路としてはグラウンドに直結した定数へと縮退してしまう。しかし、平衡三値論理ではそうはならない。

下表に示すように、平衡三値の論理積（AND）は「2つの値のうち小さい方（min）」として定義され、符号反転（NOT, `~`）は正負を反転する操作（+1 ↔ −1, 0 ↔ 0）である。入力 a が +1 や −1 のときは出力が最小値の −1 となるが、a = 0（0 V）が与えられた場合、反転した ~a も 0 となるため、0 & 0 の出力は **0（0 V）** になる。すなわち `a & ~a` は定数ではなく、a = 0 の入力に対して固有の中間電圧を出力する、物理的な実体を持った立派な三値回路である。

<img src="https://cdn.jsdelivr.net/gh/PenguinEino/tritflow@ad168eb75b9755c0694159406eb999a1a291b99b/notebook/assets/tables/and.svg" width="660" alt="and">

### 1.1 HDL記述（TritVerilog）

まず、この回路をTritFlow専用のHDL「TritVerilog」で記述する。三値の信号線は型 `trit` で宣言し、代入文 `assign` で論理積 `&` と反転 `~` を記述する。

In [ ]:
# 1-1 HDL
source = """
module first(input trit a, output trit y);
  assign y = a & ~a;
endmodule
"""

### 1.2 論理合成

HDL記述を解析し、三値スタンダードセルの回路網へと変換する。TritFlowの論理合成器は三値の論理式を三値セル（ここでは反転を行うINVと、否定論理積を行うNAND）へとダイレクトに写像する。同時に、全入力パターン（ここでは a = −1, 0, +1 の3通り）を網羅的に評価し、合成後の回路網が元のHDLと数学的に完全一致することを自動検証する。

In [ ]:
# 1-2 合成する
need("source", why="合成するHDLは、1-1に書いてあります。")
with step("HDLを読み込んで合成") as s:
    net = flow.loads(source).synthesize()
    s.done(" · ".join(f"{cell}×{n}" for cell, n in net.report()["cells"].items())
           + (" · 全入力で等価" if net.equivalence["passed"] else " · 等価でない"))
bus_picker(net)

### 1.3 配置配線

合成されたゲートのネットリストをもとに、TR-1um PDKのデザインルールに沿ってセルを自動配置し、2層の配線層（M1/M2）で結線する。セル間を水平に貫く3本の電源レール（上からVDD=+5 V、VMID=0 V、VSS=−5 V）が互いに隙間なく密着するようにセルが並べられ、信号線が短距離で配線される。

In [ ]:
# 1-3 配置配線する
need("net", why="配置配線する回路 net は、1-2で合成します。")
with step("配置配線", view=LayoutView()) as s:
    lay = net.place_and_route(output_dir=BUILD/net.top, seed=0)
    width, height = lay.stats["die_size_um"]
    s.done(f"{lay.stats['instance_count']}セル · {width:.0f} × {height:.0f} µm · 配線 {lay.stats['routed_nets']}本",
           picture=replay(lay))

### 1.4 DRCとLVS

完成したレイアウトが半導体ファブの製造基準を満たしているかを、PDK公式のKLayoutデッキを用いて検証する。設計規則違反がないかを調べるDRC（Design Rule Check）と、レイアウトから抽出したトランジスタ接続が論理合成したネットリストと一致しているかを照合するLVS（Layout Versus Schematic）を自動実行する。いずれもエラーゼロ（Pass）となることを確認してほしい。

In [ ]:
# 1-4 DRCとLVS
need("lay", why="DRCとLVSは、1-3で作ったGDSを調べます。")
with step("DRC") as s:
    drc = lay.drc()
    s.done(f"違反 {drc['violations']}件")
with step("LVS") as s:
    lvs = lay.lvs()
    s.done("一致" if lvs["matched"] else "不一致")
gds_viewer(lay.gds_path)

### 1.5 ポストレイアウトシミュレーション

レイアウトの幾何形状から配線容量を見積もり、トランジスタの実際の寸法で記述されたSPICEネットリストを生成して、ngspiceによる過渡解析を実行する。入力 a を −5 V → 0 V → +5 V → 0 V → −5 V と変化させたとき、出力の電圧が目標レベル（−5 V・0 V・+5 V）の許容判定帯（±0.5 V）に整定する様子と、各内部ノードの電圧をインタラクティブな波形で確認できる。

In [ ]:
# 1-5 ポストレイアウトシミュレーション
need("lvs", why="シミュレーションは、1-4でLVSが一致したGDSから回路を取り出して動かします。")
with step("抽出と配線容量") as s:
    extracted = lay.extract()
    signals = [n for n in extracted.capacitance["nets"] if n not in ("VDD", "VMID", "VSS")]
    s.done(f"信号{len(signals)}本に配線容量を追加")
with step("ngspice") as s:
    post = extracted.simulate(stimulus(net), load=100e-15, hold_s=1e-6, step_s=2e-9,
                              edge_s=1e-9, temperature_c=27)
    s.done("全区間で ±0.5 V の帯に入った" if post["measurements"]["passed"] else "帯に入らない区間がある")
pick_input(net, post)
transient_conditions(post, load_f=100e-15)

これで、三値HDLから製造可能なGDSの生成と動作検証までの一巡が完了した。全体のパイプラインと各段の入出力関係を振り返ると、次の図のようになる。

![fig_flow_map](https://raw.githubusercontent.com/PenguinEino/tritflow/ad168eb75b9755c0694159406eb999a1a291b99b/figures/out/fig_flow_map_ja.png)

## 2. セル

1章の回路はINVとNANDの2種類のスタンダードセルでできていた。本章ではTritFlowで用いるスタンダードセルを説明する。まず最も基本的なINVを例に、CMOSで3つの電圧レベルをどう作るかを確かめる。次に、三値の任意の論理がどのようなセルから作れるのかを示し、最後に、テープアウトした回路をどのように配置配線できるスタンダードセルにしたのかを見る。

### 2.1 INV：3つのレベルを作る

二値のINVとの違いは0 Vを出すために上下のMOSを同時にオンにし、分圧で中間のレベルを作る点にある。

TritFlowのINVはVDDとVSSの間にPMOS、拡散抵抗（PDKのデバイス名RR）2本、NMOSを直列につないだ回路で、出力Yは2本の抵抗の間からとる。

<img src="https://cdn.jsdelivr.net/gh/PenguinEino/tritflow@ad168eb75b9755c0694159406eb999a1a291b99b/notebook/assets/tables/inv.svg" width="650" alt="inv">

入力が±5 Vのときは片方のMOSだけがオンになり、Yはそちら側のレールにプルアップまたはプルダウンされる。入力が0 Vでは両方のMOSがオンになり、分圧によって0 Vが出力される。0 Vはどのレールからも供給されず、上下の抵抗の釣り合いだけで作られ、その間は静的電流が流れ続ける。


#### SPICEでのシミュレーション

テープアウトした寸法を含む90通りの寸法について、ngspiceとPDKのモデルでDC解析した結果を示す。

テープアウトした寸法は中間レベルの精度、遅延、面積、静的電流のトレードオフで決めた。TritFlowのセルは実測と照合するため、この寸法をそのまま使う。図のグリッドには、誤差と静的電流がどちらもこれより小さい寸法もある。

- **目標**：中間レベルの雑音余裕として、入力 −1〜+1 Vで出力が ±0.5 V以内に収まること。
- **ゲート長**：駆動能力と面積から、最小の1 µm。
- **PMOSとNMOSの幅**：入力0 Vでの出力は、上側のオン抵抗とRRの和 r<sub>on,P</sub> + R<sub>RP</sub> が下側の r<sub>on,N</sub> + R<sub>RN</sub> と等しいとき0 Vになる。正孔の移動度は電子より低いため、PMOSの幅をNMOSより広くとって両者のオン抵抗を近づけ、出力オフセットを小さくした。
- **RRの幅**：最小の2.8 µm。
- **RRの長さ**：テープアウトしたチップ全体の静的電流を電源配線とビアの許容電流に収めるように、30 µmとした。MOSの寸法はRRを延ばす前のまま変えていないため、入力0 Vで数十mVのオフセットが残るが、±0.5 Vの判定帯に対しては十分小さい。

採用寸法ではPMOSの幅が13.5 µm、NMOSの幅が5 µmである。PDKのゼロバイアスの抵抗式から各RRを10.835 kΩ、移動度・酸化膜厚・しきい値からMOSを線形抵抗で近似すると、r<sub>on,P</sub> = 0.492 kΩ、r<sub>on,N</sub> = 0.399 kΩとなる。RRとの比はそれぞれ22.0倍、27.2倍であり、0のレベルはMOSのばらつきよりも、同じ形状の2本の抵抗の比で決まる。上側を R<sub>u</sub> = r<sub>on,P</sub> + R<sub>RP</sub>、下側を R<sub>d</sub> = r<sub>on,N</sub> + R<sub>RN</sub> とすると、分圧から

$$V_Y = V_{SS} + (V_{DD}-V_{SS})\frac{R_d}{R_u+R_d} \simeq -0.021\ \mathrm{V},\qquad I = \frac{V_{DD}-V_{SS}}{R_u+R_d} \simeq 0.443\ \mathrm{mA}$$

となる。SPICEの −0.040 V、0.407 mAとの差は、近似がRRの電圧・基板バイアス依存とMOSの非線形性を省いていることによる。

In [ ]:
# 2-1 INVのDC解析と寸法
with step("INVのDC解析") as s:
    spice = sweep_dimensions("INV", [], workdir=BUILD/"inv")
    zero = next(r for r in spice["rows"][0]["dc"] if r["input_trits"] == [0])
    s.done(f"入力 0 V：Y = {zero['output_v']['Y']:+.3f} V · {abs(zero['rail_current_a']['VDD'])*1e3:.3f} mA")
with step("寸法の記録を照合") as s:
    grid = load_grid("INV")
    s.done(f"{len(grid['rows'])}通り · 回路・モデル・解析コードが一致")
from predictions import divider, rail_power
from tritflow import PDK_ROOT
from style import figure_css, table_css
inv_hand = divider(grid["rows"][grid["adopted_index"]]["dimensions"], PDK_ROOT)
upper, lower = inv_hand["upper"], inv_hand["lower"]
resistances = [upper["rr_ohm"], upper["ron_ohm"], lower["ron_ohm"],
               upper["total_ohm"], lower["total_ohm"]]
comparison = [("V<sub>Y</sub> (V)", inv_hand["output_v"], zero["output_v"]["Y"], "+.3f"),
              ("I (mA)", inv_hand["current_a"]*1e3, abs(zero["rail_current_a"]["VDD"])*1e3, ".3f"),
              ("P (mW)", inv_hand["power_w"]*1e3,
               rail_power(zero, grid["conditions"]["supplies_v"])*1e3, ".3f")]
display(HTML('<style>'+figure_css('.inv-estimate')+table_css('.inv-estimate')+'</style>'
    + '<div class="inv-estimate"><table><tr>'
    + ''.join(f'<th>{label}<small>kΩ</small></th>' for label in
              ('R<sub>RR</sub>', 'r<sub>on,P</sub>', 'r<sub>on,N</sub>', 'R<sub>u</sub>', 'R<sub>d</sub>'))
    + '<th>R<sub>RR</sub>/r<sub>on,P</sub></th><th>R<sub>RR</sub>/r<sub>on,N</sub></th></tr><tr>'
    + ''.join(f'<td>{r/1e3:.3f}</td>' for r in resistances)
    + f'<td>{upper["rr_ohm"]/upper["ron_ohm"]:.1f}</td><td>{lower["rr_ohm"]/lower["ron_ohm"]:.1f}</td></tr></table>'
    + '<table style="margin-top:1em"><tr><th>入力0 V</th><th>抵抗による近似</th><th>SPICE</th></tr>'
    + ''.join(f'<tr><td>{label}</td><td>{estimate:{fmt}}</td><td>{measured:{fmt}}</td></tr>'
              for label, estimate, measured, fmt in comparison)
    + '</table></div>'))
inverter(spice, grid)
dc_conditions()

### 2.2 完全性：三値の論理は何から作れるか

論理合成はHDLで書いた論理を用意したセルの組合せに置き換える作業である。どんな論理でも置き換えられるには、セルの組が**機能的に完全**である必要がある。機能的に完全とはその組のセルをつなげば、あらゆる論理関数を作れることをいう。二値では、NANDが1種類あれば完全である。

三値では以下のNANYセルと定数セルTIEPの2種類だけで機能的に完全になる。

- **NANY**：TritFlowで定義した2入力のセル。2つの入力の和を−1から+1の範囲に切り詰めてから符号を反転する。NANY(a, b) = −sat(a + b)
- **sat**：−1から+1の範囲に切り詰める関数。sat(x) = max(−1, min(+1, x))
- **TIEP・TIEZ・TIEN**：+1・0・−1を出す定数セル。名前は二値のタイセル（TIEHI・TIELO）にならった。完全性にはTIEPかTIENのどちらか1つでよい。

根拠はSłupecki（1939年）の判定法である。三値の関数の組は次の2つを満たせば完全になる。

1. すべての1入力関数（単項演算）を合成可能であること：入力 a の取り得る値（−1・0・+1 の3通り）それぞれに対して出力を独立に選べるため、1入力関数は全部で 3 × 3 × 3 = 27通り存在する。これら27種類すべてを生成できなければならない。
2. 本質的な2入力関数を少なくとも1つ生成可能であること：2つの入力 a, b の双方に依存し（片方を固定しても、もう片方の変化によって出力が変化し得る）、かつ出力として −1・0・+1 の3値すべてを取り得る2入力関数が存在すること。

条件2は、NANY自身が直ちに満たす。NANY(a, 0) = −a であるため a の変化が出力に直接反映され、b についても同様に対称である。また出力値も −1・0・+1 のすべてをとる。

条件1については入力変数 a と定数 +1 のみを出発点として、既知の関数の出力をNANYの入力に次々と再帰的に接続していき、27種類すべての単項関数が生成されることを網羅的に確認する。下表はNANYの論理段数（深さ）とともに生成される関数の推移を示している。

In [ ]:
# 2-2 NANYと +1 の定数から作れる1入力関数
from completeness import generations, surjective_and_depends_on_both, nany
found = generations(binary=(nany,), constants=(1,))
print(f"作れた1入力関数：{len(found)} / 27、最も深いもので {max(found.values())} 段")
print("NANYは2つの入力の両方に依存し、出力に−1・0・+1のすべてが現れる：", surjective_and_depends_on_both(nany))
closure_view(found)

なぜNANY単体では完全にならず、定数セルが必要なのか。NANY(0, 0) = −sat(0 + 0) = 0 なので、すべての入力が0（0 V）のとき、NANYをどれだけつないでも出力は0のままであり、0以外の値は作れない。0以外の定数が1つあれば、この制約は外れる。たとえばTIEPの +1 があれば、NANY(+1, +1) = −1、NANY(+1, −1) = 0 として残りの定数も作れるので、定数セルは1種類でよい。

### 2.3 スタンダードセルの一覧

TritFlowが提供するスタンダードセルの一覧を以下に示す。INV、NAND、NOR、NANYの4つの主要論理ゲートは、OpenSUSI-TR10で実際にテープアウトされた回路そのものである。NANDやNORは完全性の観点からは必須ではないが、回路網の論理段数を削減し、面積と遅延を最適化するために用意している。これらに加え、高ファンアウト信号を駆動するためのバッファBUF（INVの2段直列接続）と、定数セル（TIEP/TIEZ/TIEN）を備える。

各セルの回路図記号において、出力端の白丸は符号反転（NOT）を意味し、NANDとNORは二値論理でおなじみの記号形状を踏襲している。NANYは総和器（Σ）と飽和・反転を組み合わせた独自のシンボルで表記する。

In [ ]:
# 2-3 セルの真理値表
cell_tables(CELLS_DIR/"out", ROOT/"notebook/assets/schematics")

### 2.4 スタンダードセルのレイアウト

自動配置配線（P&R）を実現するためには、すべてのセルが厳格な幾何学的ルールに従って設計されていなければならない。TritFlowのセルライブラリは、行の高さ（セルハイト）を全セル共通の **132 µm**、配置配線格子ピッチを **6 µm** に統一している。

セル内部には上から順に **VDD（+5 V, 幅 8 µm）**、**VMID（0 V, 幅 4 µm）**、**VSS（−5 V, 幅 4 µm）** の3本の第1層メタル（M1）電源レールが水平に貫通している。セルを横に隙間なく突き当てて並べるだけで、電源レールとNウェルが自動的に連続して接続される。信号入力ピンおよび出力ピンは、上層の第2層メタル（M2）配線が直交してビア（V1）で無理なくアクセスできるよう、配線格子に沿ったM1ランドとして配置している。また、セルは通常配置（R0）に加えて左右反転（MY）での配置も完全に許容している。

In [ ]:
# 2-4 スタンダードセル
import json
cell_row(json.loads((CELLS_DIR/"out/library.json").read_text()), CELLS_DIR/"out")

## 3. HDLと論理合成

2章では、TritFlowのセルライブラリが任意の三値論理を網羅できる機能的完全性を備えていることを確認した。本章では三値回路専用のHDLである「TritVerilog」と、その記述をスタンダードセルの接続網へと変換する論理合成エンジンについて見ていく。TritVerilogは業界標準のVerilog HDLの構文体系を三値へと自然に拡張したものであり、直感的な文法で三値デジタル回路を設計できる。

### 3.1 値と数

信号の値（トリット）は−1・0・+1で、電圧の−5 V・0 V・+5 Vに対応する。条件として使うときは、偽・不定・真を表す。不定は物理的な0 Vに対応する有効なトリット0であり、Verilogの未定義値Xや高インピーダンスZとは異なる。

複数のトリットを束ねたベクトル（例：`trit [1:0] a;`）は、**平衡三値**（Balanced Ternary）による整数値を表す。下位から数えて第 i 桁の重みは 3<sup>i</sup> となる。数値リテラルでは値 −1 を記号 `T`（あるいは小文字 `t`）で表記する。たとえば `2'tT1` は−1·3 + 1 = −2 を表す。一般に n トリットでは −(3<sup>n</sup> − 1)/2 から +(3<sup>n</sup> − 1)/2 までの 3<sup>n</sup> 通りの整数を過不足なく表せ、2トリットなら −4 から +4 までの9通りになる。

<img src="https://cdn.jsdelivr.net/gh/PenguinEino/tritflow@ad168eb75b9755c0694159406eb999a1a291b99b/notebook/assets/tables/numbers.svg" width="566" alt="numbers">

平衡三値記数法には、二進数表現（2の補数表現など）にはない優れた特徴がある。

- **符号ビットが不要**：−1 を対称に含むため、特別な符号ビットを設けることなく、正数・ゼロ・負数を単一の形式で対称に表現できる。
- **符号反転が瞬時に完了**：数全体の符号を反転（−a）させるには、単にすべての桁の正負を反転（T ↔ 1）するだけで済む。2の補数表現のように「全ビットを反転した上で最下位ビットに1を加算する」というキャリー伝搬の遅延が一切生じない。
- **切り捨てが最も近い値への丸めになる**：各桁が −1・0・+1 なので、下位の桁を捨てたときの誤差は、残した最下位の桁の重みの半分未満に正負対称に収まる。TritVerilogの `a >> 1`（3で割って最も近い整数に丸める）も、最下位の桁を捨てるだけでこの丸めになる。

### 3.2 書き方

TritVerilogで記述できる主要な演算子と構文、およびそれらが論理合成によってどのようなセル回路網へと写像されるかを下表にまとめる。

<img src="https://cdn.jsdelivr.net/gh/PenguinEino/tritflow@ad168eb75b9755c0694159406eb999a1a291b99b/notebook/assets/tables/syntax.svg" width="640" alt="syntax">

桁数の規則はVerilogに準拠しており、代入先の桁数に合わせて演算が行われる。オペランドの桁数が代入先より短い場合は上位桁が0で埋められ、代入先に収まらない上位桁は切り捨てられる。

### 3.3 論理合成の流れ

TritFlowの論理合成エンジンは、Pythonで実装された決定論的なコンパイラであり、以下の5段階を経てネットリストを生成する。

1. **構文解析と展開**：TritVerilogのソースコードを抽象構文木（AST）へ構文解析し、モジュール階層、パラメータ（`parameter`）、および生成文（`generate`）を展開して、論理の中間表現にする。`always @*` ブロック内の手続き的代入は代入ごとに信号の値を追う静的単一代入（SSA）形式へと正規化され、`if` や `case` などの条件分岐は選択回路へと展開される。
2. **テクノロジマッピング**：演算子ごとに専用のセル構造へと置き換える。加減算はテープアウトした回路と同じ構造の全加算器・半加算器の桁上げ連鎖へ、大小比較は各桁の差を飽和させて上位桁から符号を決める回路へ、一致判定（`==`）や `case` のラベルは桁ごとの一致検出器へと写像する。選択回路は3.5で述べる。
3. **論理最適化**：定数の畳み込み、共通部分式の共有、未使用ゲートの削除を行う。さらに、次の恒等式を用いて式を簡単化する。三値の論理はブール代数ではないが、これらの恒等式は三値でも成り立つ。
   - 二重反転：−(−x) = x
   - 冪等律：min(x, x) = x、max(x, x) = x
   - 定数との min・max：min(x, +1) = x、min(x, −1) = −1、max(x, +1) = +1、max(x, −1) = x
   - 0を入れたNANY：NANY(x, 0) = −x
   - ド・モルガンの法則：min(−a, −b) = −max(a, b)、max(−a, −b) = −min(a, b)
4. **ファンアウト制限とバッファ木の挿入**：後段の入力容量が過大になると、信号の立ち上がり・立ち下がりが遅くなり、整定時間が延びる。SPICEシミュレーションにより、1つの出力で同一セルを8個並列に駆動しても正しいレベルを保てることを確認しているが、設計マージンを確保するため、論理合成では1セルあたりの最大ファンアウトを**4**に制限している。ファンアウトが4を超えるネットには、バッファセル（BUF）による木構造を自動で挿入して負荷を分散する。
5. **等価性検証**：合成後のセル回路網が元のHDL記述と等価であることを検証する。方法は3.6で述べる。

### 3.4 算術：`+` と `*`

1トリットどうしの加算では、和の取り得る範囲が (−1) + (−1) = −2 から (+1) + (+1) = +2 までとなるため、桁上げ（キャリー）が必要になる。たとえば、

1 + 1 = 2 = 3·1 + (−1)

となるため、その桁の和（Sum）は −1、上位桁への桁上げ（Carry）は +1 になる。

三値全加算器（Full Adder）は、2つの入力 a、b と下位からの桁上げ c<sub>in</sub> に対し、a + b + c<sub>in</sub> = s + 3c<sub>out</sub> を満たす回路である。TritFlowの論理合成器はHDLの `+` 演算子を、著者がテープアウトした回路と同じ構造、すなわち**NANY 11個とINV 5個**からなる全加算器へと展開する（半加算器はNANY 5個とINV 2個）。多桁の加算器はこれらを桁上げでつないで構成する。

In [ ]:
# 3-1 HDL：+ で書いた全加算器
full_add_source = """
module full_add_rtl(input trit a, b, cin, output trit sum, cout);
  assign {cout, sum} = a + b + cin;
endmodule
"""

In [ ]:
# 3-2 論理合成する
need("full_add_source", why="合成するHDLは、3-1に書いてあります。")
with step("論理合成") as s:
    full_add_rtl = flow.loads(full_add_source).synthesize()
    s.done(" · ".join(f"{cell}×{n}" for cell, n in full_add_rtl.report()["cells"].items())
           + f" · 全{full_add_rtl.equivalence['vectors']}入力で等価")
bus_picker(full_add_rtl)

一方、1トリットどうしの乗算（`*`）は単純である。積は常に −1・0・+1 のいずれかに収まり、**桁上げは発生しない**。重み w と入力 x の積はw = +1 ならば x そのもの、w = −1 ならば符号反転した −x、w = 0 ならば 0 となる。論理合成器はこの乗算をテープアウトした回路と同じ**NAND 2個、NOR 1個、INV 1個**の回路網へと展開する。

In [ ]:
# 3-3 HDL：* で書いた1トリットの積
mul_source = """
module mul(input trit w, x, output trit p);
  assign p = w * x;
endmodule
"""

In [ ]:
# 3-4 論理合成する
need("mul_source", why="合成するHDLは、3-3に書いてあります。")
with step("論理合成") as s:
    mul = flow.loads(mul_source).synthesize()
    s.done(" · ".join(f"{cell}×{n}" for cell, n in mul.report()["cells"].items())
           + f" · 全{mul.equivalence['vectors']}入力で等価")
bus_picker(mul)

### 3.5 選択：場合分けを回路にする

`if-else`、`case`、三項演算子 `?:` などの条件分岐は条件信号に応じて複数の候補から1つの信号を出力へと導く選択回路（マルチプレクサ）へと合成される。

二値論理では条件が0か1の2択であるため、選択回路は単純なAND-OR回路で構成できる。しかし三値論理では、**条件信号そのものが0（不定）を取り得る**ため、条件が0のときに何を出力すべきかを定義し、回路化する必要がある。TritFlowの論理合成器は条件式の性質に応じて2種類の選択回路を使い分ける。

以下のコードセルでは同じ機能（op = +1 なら a を、op = −1 なら b を選択する）を異なる書き方で記述した例を比較する。

In [ ]:
# 3-5 HDL：同じ選択を2通りに書く
select_sources = {
"(op == 1't1) ? a : b": """
module sel_cmp(input trit a, b, op, output trit r);
  assign r = (op == 1't1) ? a : b;
endmodule
""",
"op ? a : b": """
module sel_raw(input trit a, b, op, output trit r);
  assign r = op ? a : b;
endmodule
""",
}

**比較による条件：`op == 1't1`**　比較の結果を s と置くと、s は真（+1）か偽（−1）の2値しかとらず、0にはならない。op = 0 のときは偽となり b が選ばれる。このように条件が0にならないことが保証される場合、出力 r は二値のAND-OR選択と同じ形の次の式で構成できる。

$$r = \max\bigl(\min(s,\ a),\ \min(-s,\ b)\bigr)$$

s = +1 のときは min(+1, a) = a、min(−1, b) = −1 となり、両者の max をとると a が出力される。s = −1 のときは同様に b が出力される。min、max、符号反転は、それぞれNAND、NOR、INVで合成できる。

**値そのものによる条件：`op`**　条件として信号 `op` をそのまま指定した場合、op = 0 のケースが存在する。TritVerilogの言語仕様では、条件が0のとき、a と b の値が一致していればその値を、異なっていれば0を出力すると定めている（多トリットの値では桁ごとに比べる）。この動作を実現するには、条件の3値（−1・0・+1）それぞれに対応する入力を持つ三値の選択回路（MUX3）が必要となる。

In [ ]:
# 3-6 論理合成して比べる
need("select_sources", why="合成するHDLは、3-5に書いてあります。")
with step("論理合成") as s:
    selects = {name: flow.loads(source).synthesize() for name, source in select_sources.items()}
    s.done(" · ".join(f"{name} {net.report()['total_cells']}セル" for name, net in selects.items()))
cell_counts(selects)
for net in selects.values():
    bus_picker(net)

結果を比較すると、MUX3を必要とする2つ目の記述は、1つ目に比べてセル数が多くなる。TritFlowの論理合成器は条件式が0を取らないことを中間表現の上で確かめられた場合、1つ目の構成を使う。

### 3.6 等価性の検証

論理合成の信頼性を担保するため、TritFlowは合成を実行するたびに、元のHDLの意味と合成後のゲートネットリストが一致することを自動で検証する。HDLの振る舞い（整数と真理値の評価）と、ゲートネットリスト（各セルの真理値表を用いた論理シミュレーション）を独立した2つの経路で評価し、すべての出力を照合する。

入力が12トリット以下の場合は、全数網羅（3<sup>12</sup> = 531,441通り以下）で一致を確かめる。12トリットを超える回路は、シードを固定した10万通りの疑似乱数テストベクトルで検証する。ここまでの合成結果に表示されている「全○入力で等価」は、この検証の結果である。

### 3.7 応用例：三値重みの内積

冒頭で触れた[BitNet b1.58](https://doi.org/10.48550/arXiv.2402.17764)などの三値重みニューラルネットワークの中心となる演算は、三値重み w<sub>i</sub> ∈ {−1, 0, +1} と入力 x<sub>i</sub> との内積 y = Σ<sub>i</sub> w<sub>i</sub>x<sub>i</sub> である。BitNet b1.58の入力は8ビットだが、ここでは入力も1トリットに限定する。著者がOpenSUSI-TR10でテープアウトした回路も、この内積の1項分にあたる1トリットの積和回路（X + A × B + C<sub>in</sub> = S + 3C<sub>out</sub>）である。

ここでは、3.4で解説した乗算器と加算器を組み合わせて、4項の内積回路（モジュール名 dot4）をTritVerilogで記述し、論理合成する。各項の積 w<sub>i</sub>x<sub>i</sub> は −1〜+1、4項の和は −4〜+4 の範囲をとるため、出力 y は2トリット（重み 3<sup>1</sup>、3<sup>0</sup>）で表せる。

In [ ]:
# 3-7 HDL：三値重みの4項の内積
dot4_source = """
module dot4(input trit w0, w1, w2, w3, x0, x1, x2, x3, output trit [1:0] y);
  assign y = w0*x0 + w1*x1 + w2*x2 + w3*x3;
endmodule
"""

In [ ]:
# 3-8 論理合成する
need("dot4_source", why="合成するHDLは、3-7に書いてあります。")
with step("論理合成") as s:
    dot4 = flow.loads(dot4_source).synthesize()
    s.done(" · ".join(f"{cell}×{n}" for cell, n in dot4.report()["cells"].items())
           + f" · 全{dot4.equivalence['vectors']}入力で等価")

合成されたネットリストでは、4つの乗算部分がそれぞれNAND 2個、NOR 1個、INV 1個（計16セル）で構成され、それらの和を求める加算部分が全加算器1個と半加算器2個（NANY 21個、INV 9個）で構成されていることが確認できる。

## 4. 配置配線

3章で論理合成したセルの接続情報（ネットリスト）をもとに、TR-1um PDKの幾何学的なルールに従ってセルを配置し、2層のメタル配線で結線して、GDSを生成する。本章では3.7で合成した4項の内積回路を題材として、TritFlowの配置配線器を見ていく。

### 4.1 配置配線器を作った理由

配置配線の問題自体は信号の値によらない幾何学的な結線問題であるため、原理的には二値のEDAツール（OpenROADなど）を流用することも考えられる。実際にTR-1umでも、LibreLaneへの移植実験（[luyanaa/TR-1um](https://github.com/luyanaa/TR-1um)）が進められている。

しかし、既存の二値EDAツールをそのまま三値設計に適用するには、次の3つの技術的な障壁がある。

1. **配線層の制約（2層メタル）**：TR-1umは1 µmプロセスであり、配線に使えるメタル層は第1層（M1）と第2層（M2）の2層に限られる。2層配線では、電源供給と信号配線が同じ2層を取り合う。
2. **3本の電源レールとLibertyの限界**：すべての三値セルに、VDD（+5 V）、VMID（0 V）、VSS（−5 V）の3本の電源レールを隙間なく連続して通す必要がある。また、セルの特性を記述する標準的な形式であるLibertyは、三値の論理や遷移特性を記述できない。
3. **三値特有の静的電流**：三値のセルは0 Vを出力する間、抵抗分圧による静的電流が流れ続ける。消費電流の大きいセルが1本のレールに集中すると、電源配線の許容電流を超えてしまう。

これらの問題を解決するため、TritFlowはセル行の間に専用の配線チャネルを設けて2層で配線する、軽量の配置配線器を自前で開発した。

### 4.2 配置：焼きなまし法

セルの配置には、横方向にセルが並ぶ「行」と、配線用の通り道である「チャネル」を交互に置く構成を採用する。

1. **初期配置**：回路の入力から出力に向かって、信号の流れの順に各セルを最も幅の短い行へと詰めて配置する。
2. **焼きなまし法による改善**：初期配置を出発点として、600段の焼きなまし法で配置を段階的に改善する。各段では次の3つの操作のいずれかを確率的に実行する。評価値が下がれば採用し、上がっても温度に応じた確率で採用する。温度は段が進むにつれて下げる。
   - **セルの交換（60%）**：2つのセルの位置を入れ替える。
   - **セルの移動（30%）**：1つのセルを別の行や位置へと移す。
   - **セルの左右反転（10%）**：セルの向きを通常（R0）と左右反転（MY）で切り替え、ピンの位置を近くの配線に寄せる。
3. **評価関数と電流制約**：評価値は全ネットの端子を囲む外接矩形の半周長の和（HPWL：Half-Perimeter Wire Length）に、最も長い行の幅を小さな重みで加えたものである。さらに、各セルの静的電流を行ごとに積算し、電源レールの許容電流を超える配置候補は、評価値を無限大として棄却する。

### 4.3 配線：迷路探索

配置が完了すると、配線格子（ピッチ 6 µm）の上で信号線を配線する。基本の戦略はM1を横方向（チャネル内）、M2を縦方向（セルの上の通過を許容）とする直交2層配線である。

1. **A\*探索**：端子が多く、回路全体に広く跨がるネットから順に、A\*アルゴリズムでビア（V1）を介して最短経路で結線する。
2. **引き直し**：後から配線するネットが既存の配線にふさがれて結線できなかった場合は、すべての配線をいったん外し、失敗したネットを先頭にして残りの順序を入れ替え、最初から配線し直す。この再試行を、初回を含めて最大8回行う。
3. **チャネルの拡張**：8回の試行でも配線が完了しない場合は、配線チャネルのトラック数を増やして配線空間を広げ、再び配線を試みる（最大5回）。
4. **電源幹線の生成**：縦方向のM2電源幹線を置き、各行のM1レールへ給電する。

下のコードセルを実行すると、配置配線が実行される。実行中は配置と配線を描き、終了後は配置の改善の推移とネットを引いた順を、スライダーでたどれる図に置き換わる。

In [ ]:
# 4-1 配置配線する
need("dot4", why="配置配線する回路は、3-8で合成した内積です。")
with step("配置配線", view=LayoutView()) as s:
    dot4_layout = dot4.place_and_route(output_dir=BUILD/"dot4", seed=0)
    stats = dot4_layout.stats
    s.done(f"{stats['instance_count']}セル · {stats['row_count']}行 · {stats['die_size_um'][0]:.0f} × {stats['die_size_um'][1]:.0f} µm · "
           f"配線長の見積もり {stats['hpwl_before_um']/1e3:.1f} → {stats['hpwl_after_um']/1e3:.1f} mm · ビア {stats['via_count']}個",
           picture=replay(dot4_layout))

完成した内積回路のGDSレイアウトを以下に表示する。

In [ ]:
# 4-2 GDS
need("dot4_layout", why="表示するGDSは、4-1で作ります。")
gds_viewer(dot4_layout.gds_path)

## 5. 検証とタイミング

4章で生成した内積回路のGDSを、PDK公式の物理検証デッキで検査し、配線容量を考慮したポストレイアウトシミュレーションで動作と速さを評価する。さらに、SPICEを使わずに遅延を見積もる三値の静的タイミング解析（STA）を、SPICEと照合する。

### 5.1 DRCとLVS

完成したレイアウトが東海理化のファブの製造規則を満たしているかを、OpenSUSI TR-1um PDK公式のKLayoutデッキで検証する。

1. **DRC（Design Rule Check）**：最小配線幅、最小配線間隔、最小面積、囲み（エンクロージャ）などの幾何ルールへの違反がないかを検査する。
2. **LVS（Layout Versus Schematic）**：GDSからトランジスタの接続関係（ネットリスト）を抽出し、論理合成で得たネットリストと突き合わせる。素子のつながりだけでなく、電源レール（VDD/VMID/VSS）への接続も照合する。

In [ ]:
# 5-1 DRCとLVS
need("dot4_layout", why="検証するGDSは、4-1で作ります。")
with step("DRC") as s:
    dot4_drc = dot4_layout.drc()
    s.done(f"違反 {dot4_drc['violations']}件")
with step("LVS") as s:
    dot4_lvs = dot4_layout.lvs()
    s.done("一致" if dot4_lvs["matched"] else "不一致")

### 5.2 抽出と配線容量

現在のOpenSUSI TR-1um PDKには、寄生抽出（PEX）用の公式デッキがない。そこでTritFlowはLVSで一致したGDSの配線の形状から、各ネットの基板に対する配線容量を見積もる仕組みを備えている。

見積もりではM1とM2の図形から面積と周長を求め、次の値を仮定して配線容量を計算する。これらはPDKで較正した値ではない。

- 絶縁膜（SiO<sub>2</sub>）の比誘電率：3.9
- 基板からの高さ：M1で1 µm、M2で2 µm
- 配線端のフリンジ容量：両層共通で0.02 fF/µm

配線どうしの結合容量と配線抵抗は省略し、ネットごとの集中容量としてSPICEネットリストに加える。

In [ ]:
# 5-2 抽出と配線容量
need("dot4_lvs", why="抽出は、5-1でLVSが一致したGDSから行います。")
with step("抽出と配線容量") as s:
    dot4_extracted = dot4_layout.extract()
    dot4_wires = dot4_layout.result.net_parasitics()
    s.done(f"素子のネットリスト · 信号{len(dot4_wires['nets'])}本に配線容量を追加")
wire_table(dot4_wires)

### 5.3 三値の遅延

二値の集積回路では伝搬遅延を、出力が電源電圧の半分を横切る時刻で測ることが多い。

しかし、**この定義は三値回路では使えない**。電源電圧の中点である0 V（VMID）は、それ自体が論理値0の電圧だからである。たとえば出力が −1（−5 V）から +1（+5 V）へ遷移するとき、波形は途中で必ず0 V付近を通る。中点を横切った時刻を基準にすると、まだ遷移の途中であるにもかかわらず遷移が終わったと判定してしまう。

そこでTritFlowでは遅延を**整定時間**として定義する。

> **三値の遅延**：入力が変化を始めてから、出力が最終目標の電圧（−5 V・0 V・+5 V）の判定帯（±0.5 V）に入り、二度とそこから外れなくなるまでの時間。

下のセルでは内積回路の重みを w = (+1, −1, +1, +1) に固定し、入力 x を桁上げが起きる列で変えたときのポストレイアウトシミュレーションを実行する。これは、学習を終えた重みを固定して推論を行う使い方と同じである。波形上には、各出力で最も遅い遷移の整定時間を矢印で示す。

In [ ]:
# 5-3 ポストレイアウトシミュレーション
need("dot4_extracted", why="シミュレーションする回路は、5-2で抽出します。")
from tritflow.sim.postlayout import logical_nodeset
dot4_inputs = [p.name for p in dot4.ports if p.dir == "input"]
weights = (+1, -1, +1, +1)
activations = [(0, 0, 0, 0), (1, 0, 0, 0), (1, -1, 0, 0), (1, -1, 1, 1), (-1, 1, -1, -1), (0, 0, 0, 0),
               (0, 1, 0, 0), (-1, 1, 0, 0), (1, -1, 1, 0), (-1, 1, -1, 0), (0, 0, 0, 0)]
def operands(x):
    """The circuit's input trits for the fixed weights and the activations x, in port order."""
    values = {f"w{i}": w for i, w in enumerate(weights)} | {f"x{i}": v for i, v in enumerate(x)}
    return tuple(values[p] for p in dot4_inputs)
dot4_vectors = [operands(x) for x in activations]
with step("ngspice") as s:
    dot4_post = dot4_extracted.simulate(dot4_vectors, load=100e-15, hold_s=1e-6, step_s=5e-9, edge_s=1e-9,
                                        temperature_c=27, timeout=600, workdir=BUILD/"dot4"/"postlayout",
                                        nodeset=logical_nodeset(dot4, dot4_lvs["report"], dot4_inputs, dot4_vectors[0]))
    s.done("全区間で ±0.5 V の帯に入った" if dot4_post["measurements"]["passed"] else "帯に入らない区間がある")
display(HTML(scope_figure(dot4_post, settle=True).to_html(include_plotlyjs="cdn", full_html=False,
                                                         config={"displaylogo": False, "scrollZoom": True})))
transient_conditions(dot4_post, load_f=100e-15)

### 5.4 三値のSTA

二値のSTAで考慮すべき遷移は、立ち上がり（Rise）と立ち下がり（Fall）の2種類だけである。三値論理では論理状態が3つあるため、計6種類の状態遷移（−1 → 0、−1 → +1、0 → −1、0 → +1、+1 → −1、+1 → 0）があり、セルの伝搬遅延は遷移ごとに異なる。

TritFlowの三値STAは、セルごとに「入力遷移の種類」「入力の変化時間」「出力の負荷容量」に対する伝搬遅延と出力の変化時間をSPICEで測った特性表を参照する。

- **段ごとの伝搬遅延**：入力の中点（遷移前後の電圧の中間）と出力の中点を横切る時刻の差として求め、段ごとに足し合わせる。
- **出力での整定時間**：最終出力段では中点を横切ってから ±0.5 V の判定帯に入るまでの時間を加え、5.3で定義した整定時間とそろえる。

以下に、入力の変化時間が1 nsのときのINVの特性表を示す。

In [ ]:
# 5-4 INVの遅延の表
delay_table("INV")

INVの入力は1本なので表は6行だが、NANDやNANYなどの2入力セルは、もう一方の入力の値ごとの表と、2入力が同時に遷移する場合の表を持つ。5.3と同じ入力の列について、STAの見積もりとSPICEを比較する。

In [ ]:
# 5-5 三値のSTAとSPICEの比較
need("dot4_post", why="比べるSPICEの結果は、5-3で求めます。")
from tritflow.sta import analyze
states = [dict(zip(dot4_inputs, v)) for v in dot4_vectors]
with step("三値のSTA") as s:
    dot4_timing = analyze(dot4, vector_pairs=list(zip(states, states[1:])),
                          wire_caps={n: e["wire_capacitance_f"] for n, e in dot4_wires["nets"].items()},
                          output_load_f=100e-15, input_transition_s=1e-9)
    s.done(f"{len(dot4_timing.arrival)}出力")
timing_table(dot4_post, dot4_timing)
note("STAの条件：5-3と同じ入力の列、出力の負荷 100 fF、入力の変化時間 1 ns、配線容量は5-2の抽出値。")

## 6. 評価と応用

5章では4項の内積回路を対象に、ポストレイアウトシミュレーションと三値STAによる検証を行った。本章ではまず、内積の項数 N を変えたときの面積・遅延・静的電力を評価する。続いて、HDLの記述を変更して同じフローを実行し、最後に、MPWフレームと入出力パッドを付けた最上位レイアウトを生成する。

### 6.1 内積の規模と面積

評価対象として、三値の重みと入力による内積 y = w<sub>0</sub>x<sub>0</sub> + … + w<sub>N−1</sub>x<sub>N−1</sub> を取り上げ、項数を N = 2、4、8、16 と変える。

出力 y の桁数は−N から +N までを表せる最小の桁数（N = 2、4 で2トリット、N = 8 で3トリット、N = 16 で4トリット）とする。重みはHDLの入力として残し、シミュレーションでは (+1, −1, +1, +1) を繰り返した値に固定する。加算は3.7と同じ書き方で合成し、項数に応じた構造の変更は加えない。

下のセルでは各規模の配置配線と検証の記録を示す。面積は配置配線したダイの外形寸法から求めている。

In [ ]:
# 6-1 内積の規模と面積
from scale_views import load_scaling, scaling_table, synthesis_table
# 全掃引は長時間かかる（N=16の解析とN=32の配置配線試行を含む）。
RECOMPUTE_SCALING = False #@param {type:"boolean"}
dot_scaling = load_scaling(recompute=RECOMPUTE_SCALING)
display(synthesis_table(dot_scaling), scaling_table(dot_scaling))
for attempt in dot_scaling.get("optional_attempts", []):
    stage = "配置配線が未完了" if not attempt.get("pnr") else "検証が未完了"
    note(f"追加試行 N={attempt['n']} は{stage}のため、以下の比較には含めない。")
for row in dot_scaling["rows"]:
    options = row["conditions"].get("pnr_options", {})
    if options:
        note(f"N={row['n']}で既定値から変更した配置配線条件："
             f"行間チャネルの配線トラック {options['channel_tracks']}本、"
             f"M1の幅1 µm当たりの許容電流 {options['current_density_a_per_um']*1e3:g} mA。")
conditions = dot_scaling["conditions"]
holds = "、".join(f"N={row['n']}: {row['conditions']['hold_s']*1e6:g} µs"
                  for row in dot_scaling["rows"])
note(f"記録の条件：ngspice-{dot_scaling['provenance']['simulator']['version']}、"
     f"PDK {dot_scaling['provenance']['pdk']['qualified_commit'][:7]}、"
     f"{conditions['temperature_c']:g} °C、電源 +5 / 0 / −5 V。"
     f"出力負荷 {conditions['load_f']*1e15:g} fF、入力の変化時間 {conditions['edge_s']*1e9:g} ns、"
     f"最大時間刻み {conditions['step_s']*1e9:g} ns。各状態の保持時間は {holds}。")

### 6.2 静的電力と時間平均

三値のCMOS論理ゲートは、抵抗分圧によって0 Vを保つ状態において、入力を切り替えなくても静的電流が流れる。ただしNANYセルには、上下の直列MOS経路を遮断し、VMIDへのクランプによって0 Vを出力する状態もある。

静的電力は記録した入力列に含まれる各状態のDC動作点からSPICEで求める。時間平均の供給電力には入力を駆動する電圧源の電力を含めない。

In [ ]:
# 6-2 静的電流と電力
need("dot_scaling", why="表示する記録は、6-1で読み込みます。")
from scale_views import power_table, power_figure
power_plot = power_figure(dot_scaling)
power_plot.update_yaxes(title_text="供給電力 (mW)")
display(power_table(dot_scaling), power_plot)

### 6.3 STAとSPICE

各回路規模において、正負の桁上げの発生と解消を含む入力の列を与え、固定した重みの条件の下で、三値STAの予測遅延とSPICEの過渡遅延を比較する。4項の入力列は5章とは異なる。STAとSPICEには、同じ入力遷移、入力の変化時間、出力負荷、配線容量を与える。比較する遅延はその列で得られた出力ごとの最大整定時間である。

さらに、配線容量の有無による遅延の違いを比較する。素子の寄生容量と100 fFの出力負荷は両方に残し、見積もった配線容量だけを除いた場合と比較する。STAは全規模で、SPICEは4項で比較する。

なお、16項の回路では、一部のNANYセルの負荷容量が特性表の上限（2 pF）を超え、最大 2.862 pF に達する。このため、STAは遅延を負荷容量について線形外挿する。

In [ ]:
# 6-3 STAとSPICE
need("dot_scaling", why="比較する記録は、6-1で読み込みます。")
from scale_views import scaling_figure, layout_penalty_table
display(scaling_figure(dot_scaling), layout_penalty_table(dot_scaling))

### 6.4 HDLの変更と再実行

推論時の重みをHDLの定数として記述すると、重みに応じた乗算回路が論理合成で簡単化される。以下のセルでは3.7の内積回路の重みを (+1, −1, +1, +1) に固定して、TritVerilogを書き直している。式と入出力の宣言を変更しても、論理合成から配置配線、DRC/LVS、配線容量の見積もり、ポストレイアウトシミュレーションまでを、同じセルで続けて実行できる。

In [ ]:
# 6-4 HDLを変更する
reuse_source = """
module dot4_fixed(input trit x0, x1, x2, x3, output trit [1:0] y);
  assign y = x0 - x1 + x2 + x3;
endmodule
"""

SPICEの過渡解析では、全状態数が27以下であればすべての組合せを、それを超える場合は、ゼロ入力、全入力が −1 または +1 の状態、符号が交互になる状態、1入力だけが ±1 の状態など、代表的な11状態を選んで検証する。出力の期待値は合成した回路網からではなく、HDLの評価から求める。

In [ ]:
# 6-5 変更したHDLから全工程を実行する
need("reuse_source", why="実行するHDLは、6-4に書いてあります。")
from reuse import run
reuse_result = run(reuse_source, output_dir=BUILD/"reuse", step_s=5e-9,
                   exhaustive_limit=27, sample_size=11)

### 6.5 シャトル用の最上位レイアウト

設計した三値コアを実際のファブで製造するには、チップの外周に入出力パッドを配置し、静電気放電（ESD）保護素子を備えたMPW（Multi-Project Wafer）の標準フレームと結合する必要がある。

本節では4章で配置配線した内積回路を中心に、OpenSUSI TR-1umの標準MPWフレームと接続した最上位レイアウトを生成し、DRC、LVS、運営者のプレチェックで検証する。

ここで重要になるのが、ESD保護回路と三値の電源電圧の整合である。標準パッドのESD保護素子は、パッドの電圧がESD保護の電源（VDDとVSS）の範囲を外れると導通する。一方、三値回路の信号電圧は −5 V〜+5 V で、全体の電位差は10 Vに達する。

そこでTritFlowではフレームのESD保護の電源のうち、VSSを回路のVSS（−5 V）と共通にし、VDDには別の端子から+5 Vを与える。これにより、−5 V〜+5 V の三値信号がESD保護の電源の範囲内に収まり、標準のパッドをそのまま使える。この接続は2026年にテープアウトしたチップ「OpenSUSI-TR10」と同じである。

In [ ]:
# 6-6 MPWフレームとパッドを付けた内積回路
need("dot4_layout", why="フレームに載せるdot4は、4-1で配置配線します。")
import subprocess
from reuse import chip_view
# The organizer's MPW precheck template, pinned to one commit.
mpw_template = BUILD/"TR-1um_MPW_Sanken"
if not (mpw_template/"scripts/pre_check.py").is_file():
    subprocess.run(["git", "clone", "-q", "https://github.com/munetomo-maruyama/TR-1um_MPW_Sanken.git",
                    str(mpw_template)], check=True)
    subprocess.run(["git", "-C", str(mpw_template), "checkout", "-q",
                    "b3dab389ddaac1056c93bf5704230c6d2f52d5b2"], check=True)
# The precheck runs in a child Python; give it the same KLayout module as this kernel.
import os, pya
os.environ["PYTHONPATH"] = os.pathsep.join(filter(None, [str(Path(pya.__file__).resolve().parents[1]),
                                                         os.environ.get("PYTHONPATH")]))
with step("MPWフレームに載せて検証") as s:
    dot4_chip = dot4_layout.result.to_chip(frame="OSS_FRAME", pdk_root=dot4_layout.settings.pdk,
                                           klayout_binary=dot4_layout.settings.klayout,
                                           output_dir=BUILD/"dot4_chip", mpw_template=mpw_template,
                                           force_signoff=True)
    s.done(f"2500 × 2500 µm · パッド{len(dot4_chip.data['pad_map'])}個 · "
           + ("全検査に合格" if dot4_chip.report["clean"] else "不合格の検査がある"))
dot4_chip_report = chip_view(dot4_chip, dot4)

### 6.6 結論

本ノートブックでは、日本の商用半導体ファブが公開したオープンソースPDK「OpenSUSI TR-1um」を対象に、三値集積回路のための自動設計フロー「TritFlow」を実証した。

- **三値専用HDLからGDSまでのパイプライン**：TritVerilogによる記述から、三値の恒等式による簡単化を含む論理合成、2層メタルの配置配線、GDS出力、PDK公式デッキによるDRC/LVS、配線容量の見積もり、ポストレイアウトシミュレーションまでを一気通貫で実行する。
- **三値のタイミングモデルとSTA**：6種類の状態遷移と整定時間に基づいて遅延を定義し、SPICEを使わずに遅延を見積もる三値の静的タイミング解析を実現して、ポストレイアウトSPICEと比較した。
- **実測による実証**：スタンダードセルはテープアウトした回路と回路・素子寸法が同じであり、2027年1月に届くチップの実測結果と直接照合できる。さらに、MPWシャトル用のフルチップレイアウトを自動生成するため、TritFlowで設計した回路をそのままシャトルに提出して製造し、実測で確かめられる。

TritFlowはBitNetに代表される三値の計算に応用されうる平衡三値論理回路を学ぶための最初の一歩になる。ソースコード、セルライブラリ、設計フローはすべてApache-2.0で公開しており、三値の回路に関心を持つ研究者や学生が自ら設計を試せる出発点となる。そして、TR-1umとOpenSUSIのシャトルという日本のオープンソース半導体の基盤の上で、設計から製造、実測までを一巡させる本取り組みが、日本のオープンソース半導体設計の広がりを後押しすることを期待する。

## 7. Playground

本章は、読者が自ら三値回路を設計し、TritFlowのフロー全体を試せる実験場である。

下のコードセル内のTritVerilogのソースコードを書き換えて実行するだけで、論理合成、配置配線、GDSの生成、DRC/LVS、配線容量の見積もり、ポストレイアウトシミュレーションの全工程が自動で実行され、回路図、レイアウト、波形が表示される。

初期状態では、桁上げを伴う2トリットの加算器（`add2`）が書かれている。次のような回路を自由に書いて、三値回路の設計を試してほしい。

- **選択回路**：`op == 1't1 ? a : b` や `case` 文を用いたセレクタ
- **三値のALU**：加算 `+`、減算 `-`、論理積 `&`、論理和 `|`、符号反転 `~` を組み合わせた回路
- **独自の三値回路**：NANYなどのセルを直接置いた回路
- **内積回路**：項数や桁数を変えた内積回路

In [ ]:
# 7-1 Playground
playground_source = """
module add2(input trit [1:0] a, b, output trit [2:0] s);
  assign s = a + b;
endmodule
"""
from reuse import run
playground = run(playground_source, output_dir=BUILD/"playground", step_s=5e-9,
                 exhaustive_limit=27, sample_size=11)

## 参考文献

1. Shuming Ma et al., “The Era of 1-bit LLMs: All Large Language Models are in 1.58 Bits,” arXiv:2402.17764, 2024. [doi:10.48550/arXiv.2402.17764](https://doi.org/10.48550/arXiv.2402.17764).
2. Sheng Lin, Yong-Bin Kim, and Fabrizio Lombardi, “CNTFET-Based Design of Ternary Logic Gates and Arithmetic Circuits,” *IEEE Transactions on Nanotechnology*, 10(2), pp. 217–225, 2011. [doi:10.1109/TNANO.2009.2036845](https://doi.org/10.1109/TNANO.2009.2036845).
3. Xiao-Yuan Wang et al., “High-Density Memristor-CMOS Ternary Logic Family,” *IEEE Transactions on Circuits and Systems I: Regular Papers*, 68(1), pp. 264–274, 2021. [doi:10.1109/TCSI.2020.3027693](https://doi.org/10.1109/TCSI.2020.3027693).
4. Jae Won Jeong et al., “Tunnelling-based ternary metal–oxide–semiconductor technology,” *Nature Electronics*, 2(7), pp. 307–312, 2019. [doi:10.1038/s41928-019-0272-8](https://doi.org/10.1038/s41928-019-0272-8).
5. Hanmok Park, Seonghoon Kim, Sunmean Kim, and Taigon Song, “An RTL-Based General Synthesis Methodology for Device-Independent Ternary Logic Circuits,” *IEEE Access*, 13, pp. 163648–163660, 2025. [doi:10.1109/ACCESS.2025.3597293](https://doi.org/10.1109/ACCESS.2025.3597293).
6. Steven Bos and contributors, *MixedRadixCircuitSynthesis (MRCS)*, source snapshot `0865cee`, 2023. [Repository](https://github.com/aiunderstand/MixedRadixCircuitSynthesis/tree/0865cee1f49fe9a803b4910b570f4e96e514c61b).
7. Mohnish Pai, “Ankhdjet: An Open-Source Compiler for Mask-Programmed Ternary Compute-in-ROM on an Open PDK,” arXiv:2608.26206, 2026. [doi:10.48550/arXiv.2608.26206](https://doi.org/10.48550/arXiv.2608.26206).
8. Jerzy Słupecki, “Kryterium pełności wielowartościowych systemów logiki zdań,” *Comptes Rendus des Séances de la Société des Sciences et des Lettres de Varsovie, Classe III*, 32, pp. 102–109, 1939. [原文](https://rcin.org.pl/Content/50652/PDF/WA35_4821_cz167-r1939-t32_Spraw-TNW-wIII-art1.pdf#page=110).
9. Jean W. Butler, “On complete and independent sets of operations in finite algebras,” *Pacific Journal of Mathematics*, 10(4), pp. 1169–1179, 1960. [本文](https://msp.org/pjm/1960/10-4/pjm-v10-n4-p10-s.pdf).
10. Stephen Cole Kleene, *Introduction to Metamathematics*, North-Holland / P. Noordhoff, 1952, §64. [書誌情報](https://cir.nii.ac.jp/crid/1971712334708500134).
11. Francesco Paoli and Gavin St. John, “Editorial Introduction,” *Studia Logica*, 112, pp. 1201–1214, 2024. [doi:10.1007/s11225-024-10147-9](https://doi.org/10.1007/s11225-024-10147-9).
12. OpenSUSI, Tokai Rika, and contributors, *TR-1um Open Source PDK project*, commit `5d963db`, 2026. [使用したPDK](https://github.com/OpenSUSI/TR-1um/tree/5d963db9d44e192d37a0489c208bfccf2e2f690c).
13. Robin Lu and contributors, *TR-1um: an experiment to port TR-1um and OpenSUSI MPW template to LibreLane + ALIGN*, source snapshot `b14b3f5`, 2026. [Repository](https://github.com/luyanaa/TR-1um/tree/b14b3f5cca671fd86804e3cca434a0a728c73642).
14. Einosuke Okazaki, *平衡3値論理回路 — OpenSUSI TR-1um (opensusi-tr10-balanced-ternary-logic)*, source snapshot `8a60b97`, 2026. [元の回路と提出資料](https://github.com/PenguinEino/opensusi-tr10-balanced-ternary-logic/tree/8a60b97868cad008ff21fefca5da13dcd06063f8).
15. 一般社団法人OpenSUSI, “OpenSUSI-MPW第1回募集を完了 — 企業・大学・高専・高校・個人から49枠の申し込み,” 2026年10月16日. [発表](https://prtimes.jp/main/html/rd/p/000000014.000182004.html).
16. Matthias Köfferlein and the KLayout contributors, *KLayout Layout Viewer and Editor*, version 0.30.6, 2026. [Release](https://github.com/KLayout/klayout/releases/tag/v0.30.6).
17. Holger Vogt, Giles Atkinson, and Paolo Nenzi, *Ngspice User's Manual*, version 42, 2023. [Manual](https://sourceforge.net/projects/ngspice/files/ng-spice-rework/old-releases/42/ngspice-42-manual.pdf/download).
18. ISHI会, “2025年12月イベント「東海理化シャトルのオープンPDK解説会」の参加者募集,” 2025年10月30日. [発表](https://ishi-kai.org/information/seminar/2025/10/30/DecEvent_1213.html).
19. Zane Hambly, *Takahe*, source snapshot `e7123ea`, 2026. [二値で符号化したSKY130の例](https://github.com/Zaneham/takahe/blob/e7123ea60269922acb283d576678a3aabe51f4f4/designs/setun70_physical.sv).
20. Nikolay P. Brusentsov and José Ramil Alvarez, “Ternary Computers: The Setun and the Setun 70,” in *Perspectives on Soviet and Russian Computing (SoRuCom 2011)*, IFIP AICT 357, Springer, pp. 74–80, 2011. [doi:10.1007/978-3-642-22816-2_10](https://doi.org/10.1007/978-3-642-22816-2_10)